# Cutting a word out of a galaxy

This recipe carves the letters **MERA** out of a galactic disc, and then makes the opposite
picture: the same letters, this time the only material that is kept. It is a playful result, but
the thing it teaches is not: **regions in Mera are values**. You build them, combine them with
`∪ ∩ \ !`, and hand the combination to `subregion`.

*Inspired by Mingyu Hu.*

The letters are ordinary rectangles. Each one is a `Cuboid`, a letter is the union of its
rectangles, and the word is the union of its letters. Once the word is a region, cutting it out of
the disc is a single character:

```julia
subregion(gas, disc \ word)     # the disc with the word removed
subregion(gas, disc ∩ word)      # only the word
```

The same two regions are then applied to the **stars**, unchanged, because a region does not know
or care which component it is selecting from. The last section turns the word on its own into a
movie: one turn, a full flip, and the field of view breathing in and out.

**To run it on your own simulation, change two things in the next cell**: the path and the output
number. Nothing else is specific to the galaxy used here.

> This is a gallery recipe. It is not part of Mera's test suite, and it is not executed when the
> documentation is built, so its outputs are not stored here. Run it and it will produce them.

**What you need:** a RAMSES output or a converted MERA file with gas and particles, CairoMakie,
and a few minutes. The movie at the end also needs ffmpeg on your machine; everything before it
does not. The recipe reads only a slab around the disc, not the whole box.

![The word cut out of the disc, and kept instead](media/word_in_a_galaxy.png)

*Left: the word cut out of the disc. Right: the same region kept rather than dropped.*

![The same cut seen at 60 degrees](media/word_offaxis.png)

*Tilted to 60°, the letters show their depth. They are prisms cut through the slab, not paint on
a surface.*

![The same regions applied to the stars](media/word_in_stars.png)

*The same construction, selecting stars instead of gas.*

![The word turning, flipping and zooming](media/word_movie_preview.gif)

*Section 10 builds this: the word on its own, turning once, flipping right over and back, with the
field of view breathing in and out. Full resolution in
[`media/word_movie.mp4`](media/word_movie.mp4).*

| | |
|---|---|
| **Author** | Manuel Behrendt |
| **Contact** | [github.com/ManuelBehrendt](https://github.com/ManuelBehrendt) |
| **Inspired by** | Mingyu Hu |
| **Reads** | RAMSES |
| **Provenance** | `Mera v1.8.0 \| AV05CD/output_00390 \| 580.05 Myr \| L=48.0 ndim=3 lmin=6 lmax=12` |
| **Status** | contributed 2026-09-16 |

## The environment

This recipe carries its own `Project.toml`. Running the cell below uses the package versions the
recipe was written against, whatever else you have installed. `instantiate` downloads them the
first time and does nothing on later runs. See
[Reproducibility](https://manuelbehrendt.github.io/Mera.jl/stable/reproducibility/).

In [ ]:
using Pkg
Pkg.activate(@__DIR__)        # the Project.toml next to this notebook
Pkg.instantiate()             # first run only: fetches exactly those versions

## 1. The two lines you change

In [ ]:
using Mera, Printf
using CairoMakie
CairoMakie.activate!(type = "png")

# ---- change these two ----------------------------------------------------------------
const SIM    = "/path/to/your/simulation"   # a RAMSES output folder, or a MERA-file folder
const OUTPUT = 390                          # the snapshot number
# --------------------------------------------------------------------------------------

# The slab the word is carved from. Thick enough that the tilted view has something to show:
# a razor-thin disc looks like a line when you tip it.
const R_DISC = 14.0      # kpc, disc radius
const SLAB   =  4.0      # kpc, half thickness

# The word, and how big it is drawn.
const WORD       = "MERA"
const WORD_WIDTH = 22.0   # kpc, total width
const CAP_HEIGHT =  7.0   # kpc, height of a letter
const TRACKING   = 0.28   # gap between letters, as a fraction of letter width

# The stellar disc is far more centrally concentrated than the gas, so the word is drawn
# smaller for it. Same construction, different numbers.
const R_STAR      = 9.0
const STAR_WIDTH  = 13.0
const STAR_HEIGHT =  4.2

const PXSIZE_PC = 70.0            # pixel size of the finished maps
const CRANGE    = (-2.0, 2.8)     # log10 Σ in M⊙/pc²
const CRANGE_STARS = (-0.8, 1.5)

## 2. Read the slab: gas and stars

`loadall` reads whatever the snapshot holds and dispatches on what it finds: a folder of RAMSES
files goes through the RAMSES readers, a folder holding `output_NNNNN.jld2` through `loaddata`. The
line does not change when you convert your data, which is why it is used here rather than
`getinfo` plus `gethydro`.

Any keyword is passed on to each getter, so the ranges below are applied while reading. Nothing
outside the slab is ever brought into memory, which on a large run is the difference between a few
GB and a few tens of GB. `components` keeps it to the two things this recipe uses.

In [ ]:
(; hydro, particles, info) = loadall(SIM, OUTPUT;
                                    components = [:hydro, :particles],
                                    xrange = [-R_DISC, R_DISC],
                                    yrange = [-R_DISC, R_DISC],
                                    zrange = [-SLAB, SLAB],
                                    center = [:bc], range_unit = :kpc)

gas, stars = hydro, particles
@printf("%d gas cells and %d particles in the slab\n", length(gas.data), length(stars.data))

## 3. A font made of rectangles

Each letter lives in its own unit box: `x` and `y` run from 0 to 1, whatever size the letter is
finally drawn at. A glyph is just a list of rectangles in that box.

This is a stencil alphabet, the kind cut out of a sheet of metal. Every stroke is horizontal or
vertical, which is exactly what a `Cuboid` can be. Diagonals would need a tilted region, so the
sloping strokes of **M**, **R** and **A** are squared off. That keeps the whole font readable as
four lines of numbers, which is the point: you can see how a glyph is built and add your own.

In [ ]:
# each entry: (x0, x1, y0, y1) inside the letter's own 0..1 box
const GLYPHS = Dict(
    'M' => [(0.00, 0.22, 0.00, 1.00),    # left stem
            (0.78, 1.00, 0.00, 1.00),    # right stem
            (0.22, 0.78, 0.78, 1.00),    # top bar joining them
            (0.39, 0.61, 0.30, 0.78)],   # the centre wedge, squared off

    'E' => [(0.00, 0.22, 0.00, 1.00),    # spine
            (0.22, 1.00, 0.78, 1.00),    # top arm
            (0.22, 0.85, 0.39, 0.61),    # middle arm, a little shorter
            (0.22, 1.00, 0.00, 0.22)],   # bottom arm

    'R' => [(0.00, 0.22, 0.00, 1.00),    # spine
            (0.22, 0.78, 0.78, 1.00),    # top of the bowl
            (0.78, 1.00, 0.50, 1.00),    # right of the bowl
            (0.22, 0.85, 0.39, 0.61),    # waist
            (0.62, 0.84, 0.00, 0.39)],   # the leg, squared off

    'A' => [(0.00, 0.22, 0.00, 1.00),    # left stem
            (0.78, 1.00, 0.00, 1.00),    # right stem
            (0.22, 0.78, 0.78, 1.00),    # top bar
            (0.22, 0.78, 0.39, 0.61)],   # crossbar
)

## 4. From letters to one region

Now the only Mera-specific step. Each rectangle becomes a `Cuboid`, an axis-aligned box given as
`[lo, hi]` offsets from the centre. The `zrange` is deliberately taller than the slab, so every
letter cuts cleanly through it rather than floating inside it.

`reduce(∪, ...)` folds the whole list into a single region. Nothing is evaluated against the data
while this happens: a region is a description of a shape, and it meets the cells only when
`subregion` is called.

In [ ]:
"""Lay `word` out across `width` kpc and return one region covering all of it."""
function word_region(word::AbstractString; width, cap_height, tracking, z_half)
    n      = length(word)
    adv    = width / (n + (n - 1) * tracking)   # width of one letter box
    gap    = adv * tracking
    x_left = -width / 2

    boxes = Cuboid[]
    for (i, ch) in enumerate(word)
        haskey(GLYPHS, ch) || error("no glyph for '$ch'; add one to GLYPHS")
        x0 = x_left + (i - 1) * (adv + gap)
        for (rx0, rx1, ry0, ry1) in GLYPHS[ch]
            push!(boxes, Cuboid(
                xrange = [x0 + rx0 * adv,           x0 + rx1 * adv],
                yrange = [(ry0 - 0.5) * cap_height, (ry1 - 0.5) * cap_height],
                zrange = [-z_half, z_half],
                center = [:bc], range_unit = :kpc))
        end
    end
    return reduce(∪, boxes)
end

word = word_region(WORD; width = WORD_WIDTH, cap_height = CAP_HEIGHT,
                   tracking = TRACKING, z_half = 2 * SLAB)

disc = Cylinder(R_DISC, SLAB; center = [:bc], range_unit = :kpc)

## 5. Cut it out, and fill it in

Two selections from the same two regions:

- `disc \ word` is set difference: everything in the disc that is **not** in the word.
- `disc ∩ word` is intersection: only the part of the disc the letters cover.

`subregion` also takes `inverse = true`, which flips a selection without rewriting the region, so
`subregion(gas, disc ∩ word, inverse = true)` is another way to reach the first one.

In [ ]:
carved  = subregion(gas, disc \ word)    # the disc, with the word missing
letters = subregion(gas, disc ∩ word)    # only the word

@printf("disc    %9d cells\n", length(subregion(gas, disc).data))
@printf("carved  %9d cells\n", length(carved.data))
@printf("letters %9d cells\n", length(letters.data))

## 6. The two pieces have to add up

This is worth doing once, because it is the part that is easy to get wrong and easy not to notice.

A cell on the edge of a letter is partly in and partly out. Mera keeps the fraction of it that is
actually inside, so the carved disc and the letters are **complementary**: their masses sum to the
mass of the whole disc. If whole cells were kept or dropped instead, the two pieces would overlap
along every edge and the sum would come out too high.

Anything near `1e-16` here is floating-point noise, not a real difference.

In [ ]:
m_disc    = msum(subregion(gas, disc), :Msol)
m_carved  = msum(carved,  :Msol)
m_letters = msum(letters, :Msol)

@printf("disc            %.6e Msol\n", m_disc)
@printf("carved + word   %.6e Msol\n", m_carved + m_letters)
@printf("relative error  %.3e\n", abs(m_carved + m_letters - m_disc) / m_disc)

## 7. The picture

One helper, used three times below: it takes an object and two regions, projects both, and draws
them side by side on a shared colour range so the panels can be compared honestly.

**The colour range is doing real work here.** Most of the gas sits in a thin midplane, so a floor
set just below the midplane density hides everything above and below it and the galaxy looks like
a flat sheet. Reaching down to `-2.0` brings in the faint gas out of the plane, which is what makes
the slab read as something with thickness in the tilted view further down.

In [ ]:
logsd(m) = [(isfinite(v) && v > 0) ? log10(v) : NaN for v in m]

"""Project `obj` through both regions and draw the pair."""
function pair_figure(obj, inside, outside, crange, title; projkw...)
    fig = Figure(size = (1500, 780), backgroundcolor = :black)
    for (n, (region, label)) in enumerate(((outside, "disc \\ word"), (inside, "disc ∩ word")))
        pr = projection(subregion(obj, region, verbose = false), :sd, :Msol_pc2;
                        pxsize = [PXSIZE_PC, :pc], center = [:bc],
                        verbose = false, show_progress = false, projkw...)
        ax = Axis(fig[1, n]; aspect = DataAspect(), backgroundcolor = :black)
        hidedecorations!(ax); hidespines!(ax)
        e = pr.extent
        image!(ax, e[1] .. e[2], e[3] .. e[4], logsd(pr.maps[:sd]);
               colormap = :magma, colorrange = crange, nan_color = :black)
        text!(ax, e[1] + 0.5, e[4] - 0.5; text = label, color = :white,
              fontsize = 24, font = :bold, align = (:left, :top))
    end
    Label(fig[0, 1:2], title; color = RGBf(0.75, 0.78, 0.85), fontsize = 20)
    rowgap!(fig.layout, 6); colgap!(fig.layout, 6)
    return fig
end

mkpath("media")
fig = pair_figure(gas, disc ∩ word, disc \ word, CRANGE,
                  @sprintf("%s carved from the gas disc  ·  log₁₀ Σ in M⊙/pc²", WORD))
save("media/word_in_a_galaxy.png", fig)
fig

## 8. The same cut, seen from 60 degrees

Face-on, the word could be paint on a surface. It is not: the letters are prisms cut right through
the slab, and tilting the camera shows them as openings with depth, with gas piling up along the
line of sight in front of and behind them.

This is a real off-axis projection, not the face-on picture rotated. Each cell is deposited onto
the pixels its own shadow covers at this viewing angle. `fov` with a circular aperture keeps the
selection the same shape at every angle, which a box-shaped window would not.

In [ ]:
fig2 = pair_figure(gas, disc ∩ word, disc \ word, CRANGE,
                   @sprintf("the same cut at 60° inclination  ·  log₁₀ Σ in M⊙/pc²");
                   inclination = 60.0, azimuth = 30.0,
                   fov = R_DISC, fov_unit = :kpc, aperture = :circle)
save("media/word_offaxis.png", fig2)
fig2

## 9. The same regions, applied to the stars

A region carries no idea of what it is selecting from, so the identical construction works on the
particles. Only the numbers change, and only because the stellar disc is more centrally
concentrated than the gas: a word sized for the gas disc would run off the edge of the stars.

`subregion` on particles is a point-in-region test rather than a cell split, since a particle has
no volume to divide.

In [ ]:
star_word = word_region(WORD; width = STAR_WIDTH, cap_height = STAR_HEIGHT,
                        tracking = TRACKING, z_half = 2 * SLAB)
star_disc = Cylinder(R_STAR, SLAB; center = [:bc], range_unit = :kpc)

fig3 = pair_figure(stars, star_disc ∩ star_word, star_disc \ star_word, CRANGE_STARS,
                   "the same two regions, applied to the stars  ·  log₁₀ Σ in M⊙/pc²")
save("media/word_in_stars.png", fig3)
fig3

## 10. A movie: the word alone, turning, flipping and zooming

The disc is gone here. The subject is `letters` on its own, so what turns in the frame is the word
itself, cut out of real gas and carrying the structure of the gas it was cut from.

Three motions, each a function of the frame number and each periodic over the whole sequence, so
the last frame runs back into the first with no jump:

| | |
|---|---|
| **turn** | `azimuth` goes once around, 0 to 360 |
| **flip** | `inclination` goes 0 to 90 to 180 and back, right over and return |
| **zoom** | `fov` is widest face-on and tightest once the word has flipped over |

Two details that are easy to get wrong:

**Use `aperture = :circle` here, not `:square`.** The square aperture crops the frame to the
*shorter* side of the map. The word is 22 kpc wide and 7 kpc tall, so a square crop keeps the 7 and
throws the width away: you get a close-up of two middle letters instead of the word. With the
circular aperture the fov controls the framing as intended.

**Draw with `cextent`, not `extent`.** `extent` is in absolute box coordinates, while `cextent` is
the same window measured from `center`. The camera here sits at the box centre with a symmetric
fov, so `cextent` runs from `-fov` to `+fov`, which is what the axis limits expect.

If you only need one angle to sweep at a genuinely fixed field of view, `rotation_sequence` does
that in one call. It holds the fov fixed on purpose, so it is not the tool for a zoom.

In [ ]:
const NFRAMES  = 120                     # 6 seconds at 20 fps; try 12 first
const FOV_MID  = 10.0                    # kpc
const FOV_AMP  =  4.0                    # fov runs 14 kpc (wide) to 6 kpc (tight)
const MOVIE_CRANGE = (-2.5, 2.5)         # fixed, or the brightness pulses frame to frame
const MOVIE_PX = 90.0                    # coarser than PXSIZE_PC: 120 frames, so speed wins
const FFMPEG   = "ffmpeg"                # or an absolute path to the binary

turn(k) = 360.0 * k / NFRAMES
flip(k) =  90.0 * (1 - cos(2π * k / NFRAMES))
zoom(k) = FOV_MID + FOV_AMP * cos(2π * k / NFRAMES)

frames = joinpath("media", "word_frames"); mkpath(frames)
t0 = time()
for k in 0:(NFRAMES - 1)
    f  = zoom(k)
    pr = projection(letters, :sd, :Msol_pc2;
                    inclination = flip(k), azimuth = turn(k),
                    pxsize = [MOVIE_PX, :pc], fov = f, fov_unit = :kpc,
                    aperture = :circle,           # NOT :square, see above
                    center = [:bc], verbose = false, show_progress = false)

    fig = Figure(size = (900, 900), backgroundcolor = :black, figure_padding = 0)
    ax  = Axis(fig[1, 1]; aspect = DataAspect(), backgroundcolor = :black)
    hidedecorations!(ax); hidespines!(ax)
    c = pr.cextent                        # measured from `center`, so it runs -fov .. fov
    image!(ax, c[1] .. c[2], c[3] .. c[4], logsd(pr.maps[:sd]);
           colormap = :magma, colorrange = MOVIE_CRANGE, nan_color = :black)
    limits!(ax, -f, f, -f, f)             # the zoom: same picture, different window
    save(joinpath(frames, @sprintf("frame_%04d.png", k)), fig)

    el = time() - t0
    @printf("  frame %3d/%d  inc %5.1f  az %5.1f  fov %4.1f   %5.1f s, ~%4.1f min left\n",
            k + 1, NFRAMES, flip(k), turn(k), f, el, (el/(k+1)*(NFRAMES-k-1))/60)
end

In [ ]:
# ffmpeg reads the numbered frames as a sequence. `-loop 0` makes the gif repeat forever.
# The gif is for previewing in a README; the mp4 is the one worth keeping.
pat = joinpath(frames, "frame_%04d.png")
run(`$FFMPEG -y -loglevel error -framerate 20 -i $pat
     -vf "scale=900:-2:flags=lanczos" -c:v libx264 -pix_fmt yuv420p -crf 20
     -movflags +faststart media/word_movie.mp4`)

# A turbulent field dithers badly, so the gif is built through its own palette and at a lower
# frame rate. Straight gif encoding of these frames came out several times larger for no gain.
run(`$FFMPEG -y -loglevel error -i $pat
     -vf "fps=15,scale=440:-2:flags=lanczos,palettegen=max_colors=96" $(joinpath(frames, "palette.png"))`)
run(`$FFMPEG -y -loglevel error -framerate 20 -i $pat -i $(joinpath(frames, "palette.png"))
     -lavfi "fps=15,scale=440:-2:flags=lanczos[x];[x][1:v]paletteuse=dither=none"
     -loop 0 media/word_movie_preview.gif`)

@printf("mp4 %.1f MB, gif %.1f MB\n",
        filesize("media/word_movie.mp4")/1e6, filesize("media/word_movie_preview.gif")/1e6)

# The frame PNGs and the palette are intermediates: the mp4 and the gif are the
# outputs worth keeping. `media/word_frames/` is gitignored for that reason.

## The same thing, short

Once the font is defined, the whole idea is four lines.

In [ ]:
word = word_region(WORD; width = WORD_WIDTH, cap_height = CAP_HEIGHT,
                   tracking = TRACKING, z_half = 2 * SLAB)
disc = Cylinder(R_DISC, SLAB; center = [:bc], range_unit = :kpc)

projection(subregion(gas, disc \ word), :sd, :Msol_pc2, pxsize = [PXSIZE_PC, :pc])
projection(subregion(gas, disc ∩ word), :sd, :Msol_pc2, pxsize = [PXSIZE_PC, :pc])

## Attribution: run this once and paste the result

Paste the printed block into the table at the top of this notebook. The three lines write
themselves from what you actually ran, so they cannot be filled in without having run it.

In [ ]:
println("Reads        ", info.simcode)
println("Mera         ", mera_build())
println("Provenance   ", provenance_string(gas))

## The thumbnail

One picture for the card on the [gallery page](https://manuelbehrendt.github.io/Mera.jl/stable/gallery/).
The tilted view is the one that says most at card size, because the depth of the letters is
visible in it. `makethumb` ships with Mera, so there is nothing to install.

In [ ]:
makethumb("media/word_offaxis.png", "media/word_in_a_galaxy_thumb.png")

## Making it yours

- **Another word.** `word_region` takes any string whose letters are in `GLYPHS`. Add a glyph by
  writing down its rectangles in the 0..1 box; nothing else changes.
- **A heavier or lighter font.** The numbers `0.22` and `0.78` in the glyph table are the stroke
  thickness. Generate the rectangles from one constant if you want a single knob for weight.
- **Somewhere other than the middle.** Every region takes `center`, so the word can sit off to one
  side, and a `Sphere` or a `SphericalShell` can be subtracted in the same expression.
- **A tilted disc.** `Cylinder` takes an `axis`, so the disc can follow the galaxy's spin. The
  letters are axis-aligned boxes, so a genuinely tilted word needs tilted regions instead.
- **Other quantities.** Nothing here is specific to surface density. Project `:vlos`, or
  temperature, through exactly the same two selections.
- **Other components.** Section 9 uses the stars; gravity and radiative-transfer data work the
  same way.

**The wider point.** The letters are a toy, but the operation is not: `disc \ word` is the same
expression you would write to remove a satellite from a halo, to take a wedge out of a disc, or to
mask a region you do not trust.